# Quantum Algorithm Simulator & Benchmark Suite

This interactive notebook demonstrates the core capabilities of the benchmark suite:
1. **Grover's Search Algorithm** (analytical vs statevector simulation)
2. **Quantum Fourier Transform (QFT)** (Operator vs DFT matrix)
3. **Shor's Factoring Pipeline** ($N=15, 21$ via phase estimation)
4. **Variational Quantum Eigensolver (VQE)** (Transverse-Field Ising model)
5. **QAOA Max-Cut Optimization** (vs exact brute-force solver)
6. **Noisy Hardware Simulation** (Depolarizing and readout error models, error mitigation)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from qiskit.quantum_info import Statevector, Operator

from qsim.grover import grover_circuit, optimal_iterations, analytical_success_probability
from qsim.qft import qft_circuit, dft_matrix
from qsim.shor import shor_factor, order_finding_circuit, extract_order
from qsim.vqe import ising_hamiltonian, hardware_efficient_ansatz, run_vqe, exact_ground_energy
from qsim.qaoa import run_qaoa, brute_force_maxcut
from qsim.noise import depolarizing_noise_model, readout_noise_model, run_with_noise, mitigate_readout_error

## 1. Grover's Search Algorithm
Run Grover search for $n=3$ qubits with marked state $|101\rangle$ and compare against analytical formula.

In [ ]:
n = 3
marked = ['101']
k_opt = optimal_iterations(n, len(marked))
qc = grover_circuit(n, marked, iterations=k_opt)

sv = Statevector(qc)
sim_prob = sum(sv.probabilities_dict().get(bs, 0.0) for bs in marked)
theo_prob = analytical_success_probability(n, len(marked), k_opt)

print(f"Qubits: {n}, Marked State: {marked}")
print(f"Optimal Iterations: {k_opt}")
print(f"Statevector Probability: {sim_prob:.6f}")
print(f"Analytical Probability:  {theo_prob:.6f}")
print(f"Absolute Discrepancy:    {abs(sim_prob - theo_prob):.2e}")

## 2. Quantum Fourier Transform
Verify custom hand-built QFT circuit against the classical Discrete Fourier Transform matrix.

In [ ]:
n_qft = 3
qc_qft = qft_circuit(n_qft)
op_qft = Operator(qc_qft)
mat_dft = dft_matrix(n_qft)

is_equiv = op_qft.equiv(Operator(mat_dft))
norm_err = np.linalg.norm(op_qft.data - mat_dft)
print(f"QFT matches DFT matrix (up to global phase): {is_equiv}")
print(f"Norm Difference: {norm_err:.2e}")

## 3. Shor's Factoring Pipeline
Factor $N=15$ and $N=21$ using Quantum Phase Estimation order finding.

In [ ]:
p15, q15, att15 = shor_factor(15, seed=42)
print(f"Factored 15 = {p15} x {q15} in {att15} attempt(s)")

p21, q21, att21 = shor_factor(21, seed=42)
print(f"Factored 21 = {p21} x {q21} in {att21} attempt(s)")

## 4. VQE for 1D Transverse-Field Ising Model
Solve the ground state energy using Hardware-Efficient Ansatz and compare against exact classical diagonalization.

In [ ]:
n_vqe = 3
H = ising_hamiltonian(n_vqe, J=1.0, h=1.0)
exact_e = exact_ground_energy(H)

ansatz = hardware_efficient_ansatz(n_vqe, layers=2)
vqe_res = run_vqe(H, ansatz, optimizer='COBYLA', seed=42, maxiter=300)

print(f"Exact Ground Energy: {exact_e:.6f} Ha")
print(f"VQE Ground Energy:   {vqe_res['energy']:.6f} Ha")
print(f"Relative Error:      {abs(vqe_res['energy'] - exact_e) / abs(exact_e):.4%}")

## 5. QAOA for Max-Cut
Optimize cut value on a 6-node 3-regular graph and compare against exact classical brute-force solver.

In [ ]:
g = nx.random_regular_graph(3, 6, seed=42)
opt_cut, opt_strings = brute_force_maxcut(g)
print(f"Exact Maximum Cut: {opt_cut}")

for p in [1, 2, 3]:
    res = run_qaoa(g, p=p, optimizer='COBYLA', seed=42, restarts=3)
    print(f"p={p}: Expected Cut={res['expected_cut']:.4f}, Approx Ratio={res['approx_ratio']:.4f}")

## 6. Hardware Noise Simulation & Readout Error Mitigation
Demonstrate how asymmetric readout errors degrade Grover success and how calibration-matrix inversion recovers the signal.

In [ ]:
n_noise = 3
marked = ['101']
qc_noise = grover_circuit(n_noise, marked, iterations=optimal_iterations(n_noise, 1))
qc_noise.measure_all()

p_ro = 0.08
model = readout_noise_model(p_ro, p_ro)
counts = run_with_noise(qc_noise, model, shots=4096, seed=42)
raw_p = counts.get('101', 0) / 4096

mit_dict = mitigate_readout_error(counts, p_ro, p_ro, num_qubits=n_noise)
mit_p = mit_dict.get('101', 0.0)

print(f"Readout Error Probability: {p_ro}")
print(f"Raw Noisy Success Rate:    {raw_p:.4%}")
print(f"Mitigated Success Rate:    {mit_p:.4%}")